In [ ]:
import os
import pandas as pd

path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_files"
df_summary = pd.DataFrame(columns=['File', 'Mutant', 'G_wt_complex', 'G_wt_opt_apart', 'DG_wt', 'G_mut_complex', 'G_mut_opt_apart', 'DG_mut', 'DDG'])

for dir in os.listdir(path):
    ddg_path = os.path.join(path, dir, 'ddg_out')
    for file in os.listdir(ddg_path):
        if file.endswith('.ddg'):
            # print(file)
            mut = file.split('.')[0]
            with open(os.path.join(path, dir, 'mutfiles', mut + '.mutfile'), 'r') as f:
                # 将第三行的内容提取出来，去除中间的空格，得到突变信息
                mut_info = f.readlines()[2].strip().replace(' ', '')
            # read file; try to preserve original parsing but be robust if file is single-column text
            df = pd.read_csv(os.path.join(ddg_path, file), sep='\t', header=None)
            # If the file already has a numeric 4th column, use it. Otherwise extract the first float from each line.
            if df.shape[1] > 3 and pd.api.types.is_numeric_dtype(df.iloc[:, 3]):
                vals = df.iloc[:, 3].astype(float).reset_index(drop=True)
            else:
                # extract first floating point number from each row (handles lines like "... WT:  -478.838 ...")
                vals = df.iloc[:, 0].astype(str).str.extract(r'(-?\d+\.\d+)')[0].astype(float).reset_index(drop=True)

            # required indices for the calculation
            required_indices = [0, 2, 4, 6, 7, 8, 9, 11, 13, 15, 16, 17]
            if max(required_indices) >= len(vals):
                print(f"Skipping {mut_info}: not enough lines ({len(vals)})")
                continue

            # compute wild-type deltas
            g_wt_complex = round(vals.iloc[[0, 2, 4]].mean(), 3)
            g_wt_opt_apart = round(vals.iloc[[6, 7, 8]].mean(), 3)
            dg_wt = round(g_wt_complex - g_wt_opt_apart, 3)

            # compute mutant deltas
            g_mut_complex = round(vals.iloc[[9, 11, 13]].mean(), 3)
            g_mut_opt_apart = round(vals.iloc[[15, 16, 17]].mean(), 3)
            dg_mut = round(g_mut_complex - g_mut_opt_apart, 3)

            ddg = round(dg_mut - dg_wt, 3)

            # append a new row to summary
            df_summary.loc[len(df_summary)] = [
                dir,
                mut_info,
                g_wt_complex,
                g_wt_opt_apart,
                dg_wt,
                g_mut_complex,
                g_mut_opt_apart,
                dg_mut,
                ddg,
            ]
            print(f"Processed {dir} {mut_info}: DDG = {ddg}")

# write results
df_summary.to_csv(f'{path}/ddg_summary.csv', index=False)
                

Processed ddg_2khh_0001 S65A: DDG = -0.298
Processed ddg_2khh_0001 S59A: DDG = -1.325
Processed ddg_2khh_0001 G64A: DDG = -5.578
Processed ddg_2khh_0001 S62A: DDG = -0.031
Processed ddg_2khh_0001 K66A: DDG = 3.824
Processed ddg_2khh_0001 F61A: DDG = 1.665
Processed ddg_2khh_0001 G60A: DDG = -0.313
Processed ddg_2khh_0001 D58A: DDG = 5.386
Processed ddg_2khh_0001 F63A: DDG = 14.675
Processed ddg_2p1o_0010 P577A: DDG = -1.102
Processed ddg_2p1o_0010 V579A: DDG = 3.99
Processed ddg_2p1o_0010 N581A: DDG = -0.688
Processed ddg_2p1o_0010 G575A: DDG = 4.563
Processed ddg_2p1o_0010 W576A: DDG = 12.05
Processed ddg_2p1o_0010 Q572A: DDG = -0.439
Processed ddg_2p1o_0010 R580A: DDG = 2.29
Processed ddg_2p1o_0010 P578A: DDG = 0.258
Processed ddg_2p1o_0010 Y582A: DDG = -0.44
Processed ddg_2p1o_0010 K584A: DDG = 0.019
Processed ddg_2p1o_0010 V573A: DDG = -3.123
Processed ddg_2p1o_0010 R583A: DDG = -3.711
Processed ddg_2p1o_0010 V574A: DDG = 0.364
Processed ddg_1a0n_0009 G68A: DDG = -0.173
Processed d

In [2]:
import re
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 直接从汇总 CSV 读取数据
csv_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_summary.csv"
df_summary = pd.read_csv(csv_path)

# 解析 Mutant：WT氨基酸 + 位点 + 突变后氨基酸（当前应为A）
parsed = df_summary["Mutant"].str.extract(r"([A-Za-z])(\d+)([A-Za-z])$")
if parsed.isna().any().any():
    bad_rows = df_summary.loc[parsed.isna().any(axis=1), ["File", "Mutant"]]
    raise ValueError(f"Mutant 列存在无法解析的条目：\n{bad_rows.to_string(index=False)}")

df_plot = df_summary.assign(
    wt_aa=parsed[0].str.upper(),
    pos=parsed[1].astype(int),
    mut_aa=parsed[2].str.upper()
).copy()

# 仅保留丙氨酸扫描（突变后为A）
df_plot = df_plot[df_plot["mut_aa"] == "A"].copy()

if df_plot.empty:
    raise ValueError("没有检测到丙氨酸扫描数据（Mutant 末尾为 A 的记录）。")

# 每个 File 代表一个蛋白-多肽复合物，分别绘制热图
for file_name, grp in df_plot.groupby("File", sort=False):
    grp_sorted = grp.sort_values("pos", ascending=True).reset_index(drop=True)

    # 横坐标：原始氨基酸 + 位点（长度等于该组行数）
    xlabels = [f"{wt}{pos}" for wt, pos in zip(grp_sorted["wt_aa"], grp_sorted["pos"])]

    # 单行热图：每个位点一个 DDG
    heat = pd.DataFrame([grp_sorted["DDG"].to_numpy(dtype=float)], index=["A"], columns=xlabels)

    # 根据位点数量自动调整画布宽度
    fig_w = max(8, 0.6 * len(xlabels))
    plt.figure(figsize=(fig_w, 2.8))
    ax = sns.heatmap(
        heat,
        cmap="RdBu_r",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "ΔΔG"},
        center=0,
        annot=True,
        fmt=".2f",
        square=False
    )

    # 对缺失值(NaN)格子加斜线
    for i, aa in enumerate(heat.index):
        for j, col in enumerate(heat.columns):
            if pd.isna(heat.loc[aa, col]):
                ax.add_patch(
                    Rectangle(
                        (j, i), 1, 1,
                        fill=False,
                        hatch="///",
                        edgecolor="gray",
                        linewidth=0
                    )
                )

    ax.set_xlabel("WT residue (position, sorted by position)")
    ax.set_ylabel("Mutant amino acid")
    ax.set_title(f"Alanine-scan ΔΔG heatmap: {file_name}")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    # plt.show()
    plt.savefig(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_plot/{file_name}_ddg_heatmap.png', dpi=300)
    plt.close()

In [ ]:
# 检查csv文件中，ddg大于100的file，并打印出来
ddg_threshold = 100
high_ddg_files = df_summary[df_summary["DDG"] > ddg_threshold]["File"].unique()
if len(high_ddg_files) > 0:
    print(f"以下文件的 DDG 大于 {ddg_threshold}：")
    for f in high_ddg_files:
        print(f" - {f}")
else:
    print(f"没有文件的 DDG 大于 {ddg_threshold}。")

以下文件的 DDG 大于 50：
 - ddg_3wgx_0015
 - ddg_2aij_0013
 - ddg_4m91_0001
 - ddg_6f4s_0002
 - ddg_1cqg_0002


In [7]:
# 检查这些ddg大于100的文件，在minimized中查看L链是否存在cys
from Bio.PDB import PDBParser
import os
for f in high_ddg_files:
    pdb_name = f.replace("ddg_", "") + ".pdb"
    path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized", pdb_name)
    print(f"Checking {pdb_name} at {path}...")
    if os.path.exists(path):
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_name, path)
        has_cys_in_chain_L = any(
            residue.get_resname() == "CYS" and chain.get_id() == "L"
            for model in structure
            for chain in model
            for residue in chain
        )
        if has_cys_in_chain_L:
            print(f"{pdb_name} 中 L 链存在 CYS。")
        else:
            print(f"{pdb_name} 中 L 链不存在 CYS。")


Checking 3wgx_0015.pdb at /home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized/3wgx_0015.pdb...
3wgx_0015.pdb 中 L 链存在 CYS。
Checking 2aij_0013.pdb at /home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized/2aij_0013.pdb...
2aij_0013.pdb 中 L 链存在 CYS。
Checking 4m91_0001.pdb at /home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized/4m91_0001.pdb...
4m91_0001.pdb 中 L 链存在 CYS。
Checking 6f4s_0002.pdb at /home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized/6f4s_0002.pdb...
6f4s_0002.pdb 中 L 链存在 CYS。
Checking 1cqg_0002.pdb at /home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized/1cqg_0002.pdb...
1cqg_0002.pdb 中 L 链存在 CYS。


In [16]:
has_cys_in_chain_L_files = []
for file in os.listdir("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized"):
    
    pdb_path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized", file)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(file, pdb_path)
    has_cys_in_chain_L = any(
        residue.get_resname() == "CYS" and chain.get_id() == "L"
        for model in structure
        for chain in model
        for residue in chain
    )
    if has_cys_in_chain_L:
        print(f"{file} 中 L 链存在 CYS。")
        ddg_name = "ddg_" + file.split(".")[0]
        has_cys_in_chain_L_files.append(ddg_name)


1pmx_0002.pdb 中 L 链存在 CYS。
4zoz_0001.pdb 中 L 链存在 CYS。
4m91_0001.pdb 中 L 链存在 CYS。
3zrj_0009.pdb 中 L 链存在 CYS。
3wgx_0015.pdb 中 L 链存在 CYS。
6f4s_0002.pdb 中 L 链存在 CYS。
5o9u_0010.pdb 中 L 链存在 CYS。
5v2p_0018.pdb 中 L 链存在 CYS。
2aij_0013.pdb 中 L 链存在 CYS。
4yje_0008.pdb 中 L 链存在 CYS。
3p72_0001.pdb 中 L 链存在 CYS。
1cqg_0002.pdb 中 L 链存在 CYS。
5di8_0015.pdb 中 L 链存在 CYS。


In [20]:
# 读取pdb的SSBOND行，检查是否存在连接L链的SS键
for file in has_cys_in_chain_L_files:
    pdb_name = file.replace("ddg_", "") + ".pdb"
    path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized", pdb_name)
    # print(f"Checking SSBOND in {pdb_name} at {path}...")
    if os.path.exists(path):
        with open(path, 'r') as f:
            ssbond_lines = [line for line in f if line.startswith("SSBOND")]
            has_ssbond_with_L = any(
                "L" in line and "CYS" in line
                for line in ssbond_lines
            )
            if has_ssbond_with_L:
                print(f"{pdb_name} 中存在连接 L 链的 SSBOND。")

1pmx_0002.pdb 中存在连接 L 链的 SSBOND。
4m91_0001.pdb 中存在连接 L 链的 SSBOND。
3wgx_0015.pdb 中存在连接 L 链的 SSBOND。
6f4s_0002.pdb 中存在连接 L 链的 SSBOND。
2aij_0013.pdb 中存在连接 L 链的 SSBOND。
3p72_0001.pdb 中存在连接 L 链的 SSBOND。
1cqg_0002.pdb 中存在连接 L 链的 SSBOND。
5di8_0015.pdb 中存在连接 L 链的 SSBOND。


In [22]:
# 将PDB中L链存在CYS，L链存在CYS且与其他链存在SS键的文件，以及剩下的文件，分别写成三列表格
files_with_cys_in_L = set()
files_with_cys_in_L_and_ssbond = set()
for file in os.listdir("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized"):
    pdb_path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized", file)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(file, pdb_path)
    has_cys_in_chain_L = any(
        residue.get_resname() == "CYS" and chain.get_id() == "L"
        for model in structure
        for chain in model
        for residue in chain
    )
    if has_cys_in_chain_L:
        files_with_cys_in_L.add(file)
        with open(pdb_path, 'r') as f:
            ssbond_lines = [line for line in f if line.startswith("SSBOND")]
            has_ssbond_with_L = any(
                "L" in line and "CYS" in line
                for line in ssbond_lines
            )
            if has_ssbond_with_L:
                files_with_cys_in_L_and_ssbond.add(file)
files_with_cys_in_L_only = files_with_cys_in_L - files_with_cys_in_L_and_ssbond
files_without_cys_in_L = set(os.listdir("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized")) - files_with_cys_in_L

# 将结果写入表格中，分成三列的csv文件
df = pd.DataFrame({
    "File with CYS in L": pd.Series(sorted(files_with_cys_in_L_only)),
    "File with CYS in L and SSBOND": pd.Series(sorted(files_with_cys_in_L_and_ssbond)),
    "File without CYS in L": pd.Series(sorted(files_without_cys_in_L))
})
df.to_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/cys_in_L_summary.csv", index=False)


In [17]:
# 读取ddg_summary.csv，对每个file进行筛选。选择每个file中ddg大于1，且突变前不为P和G的残基，同时不考虑has_cys_in_chain_L_files中的文件。将筛选结果保存到新的csv文件中。
df_summary = pd.read_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_summary.csv')
filtered_rows = []
for file_name, group in df_summary.groupby("File"):
    if file_name in has_cys_in_chain_L_files:
        print(f"跳过 {file_name}，因为它在 minimized 中的 L 链存在 CYS。")
        continue
    for _, row in group.iterrows():
        if row["DDG"] > 1:
            wt_aa = row["Mutant"][0]  # 突变信息的第一个字符是突变前氨基酸
            if wt_aa not in ["P", "G"]:
                filtered_rows.append(row)
filtered_df = pd.DataFrame(filtered_rows)
# 首先按照file升序，随后按照DDG降序排序
filtered_df = filtered_df.sort_values(by=["File", "DDG"], ascending=[True, False])
filtered_df.to_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_filtered.csv', index=False)

跳过 ddg_1cqg_0002，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_1pmx_0002，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_2aij_0013，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_3p72_0001，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_3wgx_0015，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_3zrj_0009，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_4m91_0001，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_4yje_0008，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_4zoz_0001，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_5di8_0015，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_5o9u_0010，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_5v2p_0018，因为它在 minimized 中的 L 链存在 CYS。
跳过 ddg_6f4s_0002，因为它在 minimized 中的 L 链存在 CYS。


In [23]:
# 读取ddg_filtered.csv，统计每个file中满足条件的突变数量以及平均的ddG大小
filtered_df = pd.read_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_filtered.csv')
summary_stats = filtered_df.groupby("File").agg(
    Mutation_Count=("DDG", "count"),
    Average_DDG=("DDG", "mean")
).reset_index()
summary_stats.to_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_summary_stats.csv', index=False)

In [26]:
# 统计每种残基被突变成丙氨酸的次数，以及平均的ddG大小
filtered_df["wt_aa"] = filtered_df["Mutant"].str[0]
residue_stats = filtered_df.groupby("wt_aa").agg(
    Mutation_Count=("DDG", "count"),
    Average_DDG=("DDG", "mean"),
    Std_DDG=("DDG", "std")
).reset_index()
residue_stats = residue_stats.sort_values(by="Mutation_Count", ascending=False)
residue_stats.to_csv(f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_residue_stats.csv', index=False)